# 🎙️ VoizGen Worker (Colab GPU Engine)

Worker ini **hanya menyajikan JSON API** — seluruh antarmuka ada di web app milikmu (`web/index.html`).

### Cara pakai
1. `Runtime` → `Change runtime type` → **T4 GPU**.
2. Jalankan sel di bawah, tunggu sampai banner muncul (model ±2&ndash;4 menit).
3. Klik **TAUTAN KONEKSI** di bawah banner — web app terbuka dan langsung terhubung.
   (Tanpa tautan: salin **TIKET KONEKSI** / **KODE AKSES** ke web app secara manual.)
4. Biarkan sel ini tetap berjalan selama dipakai. `Ctrl+M I` (interrupt) untuk mematikan worker.

Endpoint: `GET /api/health`, `POST /api/generate` (multipart), auth `Authorization: Bearer <token>`.


In [ ]:
#@title 🎙️ VoizGen Worker Engine — klik Play
#@markdown Kosongkan ketiga kolom di bawah bila web app dibuka langsung (mode tiket).
#@markdown Isi dua yang pertama bila web app di-deploy sebagai VoizGen Gateway di
#@markdown Cloudflare Workers — outputnya jadi **Kode Akses** + tautan sekali klik.
GATEWAY_URL = ""  #@param {type:"string"}
REGISTER_KEY = ""  #@param {type:"string"}
#@markdown Alamat web app-mu (mode tiket, mis. `https://voizgen.pages.dev`). Diisi →
#@markdown output memuat tautan sekali klik; dikosongkan → tiket saja untuk ditempel.
WEB_APP_URL = ""  #@param {type:"string"}

!pip install -q omnivoice pycloudflared fastapi "uvicorn[standard]" python-multipart

import base64
import io
import os
import re
import secrets
import tempfile
import threading
import time
import urllib.parse
import wave

import numpy as np
import requests
import torch
import uvicorn
from fastapi import Depends, FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, Response
from pycloudflared import try_cloudflare

from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.common import get_best_device
from omnivoice.utils.lang_map import LANG_NAMES, lang_display_name

PORT = int(os.environ.get("VOIZGEN_PORT", 7860))
MODEL_ID = os.environ.get("VOIZGEN_MODEL", "k2-fsa/OmniVoice")
TOKEN = os.environ.get("VOIZGEN_TOKEN") or secrets.token_urlsafe(18)

# ---------------------------------------------------------------- model
device = get_best_device()
IS_GPU = str(device) != "cpu"
# fp16 hanya di GPU: banyak operator fp16 tidak didukung / sangat lambat di CPU.
DTYPE = torch.float16 if IS_GPU else torch.float32

if not IS_GPU:
    print("\033[1;31m[PERINGATAN] Runtime berjalan di CPU. Ubah ke GPU T4: "
          "Runtime > Change runtime type > T4 GPU.\033[0m", flush=True)

print(f"[VoizGen] Memuat {MODEL_ID} pada {device} ({DTYPE})...", flush=True)
model = OmniVoice.from_pretrained(MODEL_ID, device_map=device, dtype=DTYPE, load_asr=True)
LANGUAGES = sorted(lang_display_name(name) for name in LANG_NAMES)
GPU_LOCK = threading.Lock()  # satu inferensi sekaligus: VRAM T4 terbatas

# Kosakata instruct diambil dari pustaka agar tidak pernah beda dengan model.
# Bila nama internalnya berubah di versi baru, daftar dikirim kosong dan
# validasi sepenuhnya jatuh ke worker (400 dengan pesan jelas).
try:
    from omnivoice.utils.voice_design import _INSTRUCT_VALID_EN, _INSTRUCT_VALID_ZH

    INSTRUCT_ATTRIBUTES = sorted(_INSTRUCT_VALID_EN) + sorted(_INSTRUCT_VALID_ZH)
except Exception:
    INSTRUCT_ATTRIBUTES = []

# Tag non-verbal yang benar-benar dilatih (docs/README OmniVoice).
NONVERBAL_TAGS = [
    "laughter", "sigh", "confirmation-en", "question-en", "question-ah",
    "question-oh", "question-ei", "question-yi", "surprise-ah", "surprise-oh",
    "surprise-wa", "surprise-yo", "dissatisfaction-hnn",
]

# Arahan naskah milik worker ini. Tiap arahan dipetakan ke kontrol model yang
# nyata (jeda sunyi, faktor speed, atribut whisper) — bukan emosi karangan.
DIRECTIVES = ["pause", "pause:1.5", "slow", "fast", "whisper", "normal"]
DIRECTIVE_SPEED = {"slow": 0.85, "lambat": 0.85, "fast": 1.15, "cepat": 1.15}
DIRECTIVE_WHISPER = {"whisper", "berbisik"}
DIRECTIVE_RESET = {"normal", "reset"}
DEFAULT_PAUSE = 0.4
MAX_PAUSE = 5.0
MAX_SEGMENTS = 40

_BRACKET_RE = re.compile(r"\[([^\[\]]*)\]")
_PAUSE_RE = re.compile(r"^(?:pause|jeda)(?:\s*[:=]\s*([0-9]*\.?[0-9]+))?$")
_CMU_RE = re.compile(r"^[A-Z0-9 ]+$")


def parse_script(raw: str):
    """Pecah naskah berarah menjadi daftar segmen + jeda.

    Segmen: {"text", "speed_mul", "whisper", "pause_after"}.
    Tag non-verbal resmi dan koreksi pelafalan CMU dibiarkan utuh di dalam teks
    karena model memang memahaminya. Tag lain dibuang dan dilaporkan.
    """
    segments: list[dict] = []
    dropped: list[str] = []
    speed_mul = 1.0
    whisper = False
    buffer: list[str] = []
    pos = 0

    def flush(pause_after: float = 0.0) -> None:
        nonlocal buffer
        body = " ".join("".join(buffer).split())
        if body:
            segments.append({
                "text": body,
                "speed_mul": speed_mul,
                "whisper": whisper,
                "pause_after": pause_after,
            })
        elif pause_after and segments:
            segments[-1]["pause_after"] = min(
                segments[-1]["pause_after"] + pause_after, MAX_PAUSE
            )
        buffer = []

    for match in _BRACKET_RE.finditer(raw):
        inner = match.group(1).strip()
        low = inner.lower()
        # dipahami model: biarkan menempel di teks (pos tidak digeser)
        if low in NONVERBAL_TAGS or _CMU_RE.match(inner):
            continue

        buffer.append(raw[pos:match.start()])
        pos = match.end()

        pause = _PAUSE_RE.match(low)
        if pause:
            seconds = float(pause.group(1)) if pause.group(1) else DEFAULT_PAUSE
            flush(min(seconds, MAX_PAUSE))
        elif low in DIRECTIVE_SPEED:
            flush()
            speed_mul = DIRECTIVE_SPEED[low]
        elif low in DIRECTIVE_WHISPER:
            flush()
            whisper = True
        elif low in DIRECTIVE_RESET:
            flush()
            speed_mul, whisper = 1.0, False
        else:
            dropped.append(inner)

    buffer.append(raw[pos:])
    flush()
    return segments, dropped


def join_segments(waves, segments, sampling_rate: int) -> np.ndarray:
    """Rangkai audio antar segmen dengan sunyi sesuai [pause]."""
    pieces = []
    for wave_data, segment in zip(waves, segments):
        pieces.append(wave_data)
        gap = segment["pause_after"]
        if gap > 0:
            pieces.append(np.zeros(int(gap * sampling_rate), dtype=wave_data.dtype))
    return pieces[0] if len(pieces) == 1 else np.concatenate(pieces)

# ---------------------------------------------------------------- helper
def to_wav(samples: np.ndarray, sampling_rate: int) -> bytes:
    """float32 [-1, 1] -> WAV PCM 16-bit mono. Clip wajib: >1.0 akan wrap-around."""
    pcm = (np.clip(samples, -1.0, 1.0) * 32767.0).astype("<i2")
    buf = io.BytesIO()
    with wave.open(buf, "wb") as wav:
        wav.setnchannels(1)
        wav.setsampwidth(2)
        wav.setframerate(int(sampling_rate))
        wav.writeframes(pcm.tobytes())
    return buf.getvalue()


def require_token(authorization: str = Header(default="")) -> None:
    supplied = authorization.removeprefix("Bearer ").strip()
    if not secrets.compare_digest(supplied, TOKEN):
        raise HTTPException(status_code=401, detail="Token tidak valid.")


# ---------------------------------------------------------------- api
app = FastAPI(title="VoizGen Worker", docs_url=None, redoc_url=None)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["*"],
    expose_headers=["X-Elapsed", "X-Segments", "X-Notes"],
    max_age=86400,
)


@app.get("/api/health", dependencies=[Depends(require_token)])
def health():
    return {
        "ok": True,
        "engine": "omnivoice",
        "device": str(device),
        "gpu": IS_GPU,
        "dtype": str(DTYPE).replace("torch.", ""),
        "model": MODEL_ID,
        "sampling_rate": model.sampling_rate,
        "languages": LANGUAGES,
        "instruct_attributes": INSTRUCT_ATTRIBUTES,
        "instruct_freeform": False,
        # Nilai bawaan yang masuk akal untuk mesin ini; UI mengikutinya saat terhubung.
        "defaults": {"num_step": 32, "guidance_scale": 2.0},
        "nonverbal_tags": NONVERBAL_TAGS,
        "directives": DIRECTIVES,
        "busy": GPU_LOCK.locked(),
    }


@app.post("/api/generate", dependencies=[Depends(require_token)])
def generate(
    mode: str = Form("design"),
    text: str = Form(...),
    language: str = Form(""),
    instruct: str = Form(""),
    ref_text: str = Form(""),
    num_step: int = Form(32),
    guidance_scale: float = Form(2.0),
    denoise: bool = Form(True),
    speed: float = Form(1.0),
    duration: float = Form(0.0),
    preprocess_prompt: bool = Form(True),
    postprocess_output: bool = Form(True),
    ref_audio: UploadFile | None = File(None),
):
    if not text.strip():
        raise HTTPException(status_code=400, detail="Naskah teks kosong.")
    if mode not in ("clone", "design"):
        raise HTTPException(status_code=400, detail=f"Mode tidak dikenal: {mode}")

    segments, dropped = parse_script(text)
    if not segments:
        raise HTTPException(status_code=400, detail="Naskah hanya berisi tag, tidak ada teks untuk diucapkan.")
    if len(segments) > MAX_SEGMENTS:
        raise HTTPException(
            status_code=400,
            detail=f"Naskah terpecah menjadi {len(segments)} segmen (maksimum {MAX_SEGMENTS}). Kurangi jumlah arahan atau pecah naskah.",
        )

    notes = []
    if dropped:
        notes.append("tag tidak didukung diabaikan: " + ", ".join(f"[{d}]" for d in dict.fromkeys(dropped)))

    base_instruct = instruct.strip()
    instructs = []
    for segment in segments:
        parts = [base_instruct] if base_instruct else []
        if segment["whisper"] and "whisper" not in base_instruct.lower():
            parts.append("whisper")
        instructs.append(", ".join(parts) or None)

    # 1.0 = netral. None per elemen dilarang: saat estimasi durasi, omnivoice
    # membaca speed[i] mentah dan menguji `speed > 0` sebelum None-nya
    # dinormalkan (omnivoice/models/omnivoice.py:1128 lalu :1179) -> TypeError.
    speeds = [speed * segment["speed_mul"] for segment in segments]

    kwargs = {
        "text": [segment["text"] for segment in segments],
        "language": language.strip() or None,
        "generation_config": OmniVoiceGenerationConfig(
            num_step=max(1, num_step),
            guidance_scale=guidance_scale,
            denoise=denoise,
            preprocess_prompt=preprocess_prompt,
            postprocess_output=postprocess_output,
        ),
    }
    if any(value != 1.0 for value in speeds):
        kwargs["speed"] = speeds
    if any(value is not None for value in instructs):
        kwargs["instruct"] = instructs
    if duration > 0:
        if len(segments) > 1:
            notes.append("durasi tetap diabaikan karena naskah terpecah menjadi beberapa segmen")
        else:
            kwargs["duration"] = duration

    ref_path = None
    try:
        if mode == "clone":
            if ref_audio is None:
                raise HTTPException(status_code=400, detail="Audio referensi wajib untuk voice clone.")
            suffix = os.path.splitext(ref_audio.filename or "")[1] or ".wav"
            with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as handle:
                handle.write(ref_audio.file.read())
                ref_path = handle.name
            kwargs["voice_clone_prompt"] = model.create_voice_clone_prompt(
                ref_audio=ref_path,
                ref_text=ref_text.strip() or None,
            )

        started = time.perf_counter()
        try:
            with GPU_LOCK:
                audio = model.generate(**kwargs)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            # JSONResponse, bukan raise: 500 dari luar CORSMiddleware kehilangan header CORS.
            return JSONResponse({"detail": "VRAM habis. Kurangi panjang teks atau Inference Steps."}, 503)
        except ValueError as exc:
            # Instruct di luar kosakata: pesan pustaka panjang, ambil bagian yang berguna.
            head = [line.strip() for line in str(exc).splitlines() if line.strip()][:4]
            return JSONResponse({"detail": " ".join(head)}, 400)
        except Exception as exc:
            return JSONResponse({"detail": f"{type(exc).__name__}: {exc}"}, 500)
        elapsed = time.perf_counter() - started
    finally:
        if ref_path:
            os.unlink(ref_path)

    headers = {"X-Elapsed": f"{elapsed:.2f}", "X-Segments": str(len(segments))}
    if notes:
        headers["X-Notes"] = urllib.parse.quote("; ".join(notes))

    return Response(
        content=to_wav(join_segments(audio, segments, model.sampling_rate), model.sampling_rate),
        media_type="audio/wav",
        headers=headers,
    )


# ---------------------------------------------------------------- gateway
GATEWAY = (GATEWAY_URL or os.environ.get("VOIZGEN_GATEWAY", "")).strip().rstrip("/")
REG_KEY = (REGISTER_KEY or os.environ.get("VOIZGEN_REGISTER_KEY", "")).strip()
HEARTBEAT_EVERY = 300  # gateway memberi TTL 12 jam; refresh tiap 5 menit sudah aman


def gateway_call(path: str, payload: dict, timeout: int = 15) -> dict:
    resp = requests.post(
        f"{GATEWAY}{path}",
        json=payload,
        headers={"X-Register-Key": REG_KEY},
        timeout=timeout,
    )
    if resp.status_code >= 400:
        raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:200]}")
    return resp.json()


def gateway_heartbeat_loop(code: str) -> None:
    """Selama sel berjalan, perpanjang TTL kode di gateway."""
    while True:
        time.sleep(HEARTBEAT_EVERY)
        try:
            gateway_call("/api/heartbeat", {"code": code}, timeout=10)
        except Exception as exc:
            print(f"[VoizGen] Heartbeat gagal: {exc}", flush=True)


# ---------------------------------------------------------------- boot
server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
while not server.started:
    time.sleep(0.2)

tunnel = try_cloudflare(port=PORT)
public_url = str(getattr(tunnel, "tunnel", None) or getattr(tunnel, "url", None) or tunnel).rstrip("/")

# Tiket = URL + token dalam satu string, supaya cukup satu kali tempel di web app.
ticket = "voizgen_" + base64.urlsafe_b64encode(f"{public_url}|{TOKEN}".encode()).decode().rstrip("=")

access_code = None
if GATEWAY and REG_KEY:
    try:
        result = gateway_call("/api/register", {"tunnel_url": public_url, "worker_token": TOKEN})
        access_code = result.get("pretty") or result["code"]
        threading.Thread(
            target=gateway_heartbeat_loop, args=(result["code"],), daemon=True
        ).start()
    except Exception as exc:
        print(f"\033[1;31m[VoizGen] Registrasi ke gateway gagal: {exc}\033[0m", flush=True)
        print("Pakai TIKET KONEKSI di bawah sebagai gantinya.", flush=True)
elif GATEWAY or REG_KEY:
    print("\033[1;33m[VoizGen] GATEWAY_URL dan REGISTER_KEY harus diisi berdua.\033[0m", flush=True)

# Tautan sekali klik. Kredensial ditaruh di fragment (#), bukan query: fragment
# tidak pernah dikirim ke server, jadi token/kode tak masuk log Cloudflare.
APP = (GATEWAY if access_code else (WEB_APP_URL or os.environ.get("VOIZGEN_APP", "")).strip()).rstrip("/")
if APP and access_code:
    quick_link = f"{APP}/#c={access_code}"
elif APP:
    quick_link = f"{APP}/#w={ticket}"
else:
    quick_link = ""

header = f'  VoizGen Worker Engine  ·  {"GPU " + str(device) if IS_GPU else "CPU (lambat)"}'
link_block = f"""  TAUTAN KONEKSI (klik — web app langsung terhubung):

  \033[1;36m{quick_link}\033[0m

------------------------------------------------------------
""" if quick_link else ""
if access_code:
    body = f"""{link_block}  KODE AKSES (bila menempel manual):

  \033[1;32m{access_code}\033[0m

  Web app  : \033[1;36m{GATEWAY}\033[0m
  Kode berlaku 12 jam selama sel ini berjalan."""
else:
    body = f"""{link_block}  TIKET KONEKSI (bila menempel manual):

  \033[1;32m{ticket}\033[0m

------------------------------------------------------------
  Manual, bila perlu:
  WORKER URL : \033[1;36m{public_url}\033[0m
  TOKEN      : \033[1;36m{TOKEN}\033[0m"""

print(f"""
============================================================
{header}
============================================================
{body}
============================================================
  Sel ini harus tetap berjalan. Interrupt untuk mematikan.
============================================================
""", flush=True)

# Colab tidak selalu menjadikan URL di stdout bisa diklik; anchor HTML selalu bisa.
if quick_link:
    try:
        from IPython.display import HTML, display

        display(HTML(
            f'<a href="{quick_link}" target="_blank" rel="noopener" style="display:inline-block;'
            'padding:10px 18px;margin:6px 0;border-radius:8px;background:#2563eb;color:#fff;'
            'font:600 14px system-ui,sans-serif;text-decoration:none">▶ Buka VoizGen Studio &amp; hubungkan</a>'
        ))
    except ImportError:
        pass

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    server.should_exit = True
    if access_code and GATEWAY and REG_KEY:
        try:
            gateway_call("/api/deregister", {"code": access_code.replace("-", "")}, timeout=10)
            print("[VoizGen] Kode akses dicabut dari gateway.")
        except Exception as exc:
            print(f"[VoizGen] Gagal mencabut kode: {exc}")
    print("[VoizGen] Worker dimatikan.")
